# Lab 6: Building a Customer-Facing Frontend Application

## Overview

In the previous labs, we've built a comprehensive Customer Support Agent with memory, shared tools, production-grade deployment, and online evaluation. With them we showcased the capabilities of AgentCore services to move an agentic use case from prototype to production. You can now invoke your agent runtime from any application. On real world applications, customers expect a user interface to be available. Now it's time to look at the user-friendly frontend that customers can actually use to interact with our agent.

**Workshop Journey:**
- **Lab 1 (Done)**: Create Agent Prototype - Built a functional customer support agent
- **Lab 2 (Done)**: Enhance with Memory - Added conversation context and personalization
- **Lab 3 (Done)**: Scale with Gateway & Identity - Shared tools across agents securely
- **Lab 4 (Done)**: Deploy to Production - Used AgentCore Runtime with observability
- **Lab 5 (Done)**: Evaluate Agent Performance - Monitor quality with online evaluations
- **Lab 6 (Current)**: Explore the User Interface - A customer-facing application

Unlike the previous labs, this one isn't about creating new AWS resources from notebook cells — the frontend is a standalone application that already lives in this repository, fully built and ready to run. This lab walks through what it does and how to launch it.

The application is a **Next.js + Tailwind CSS** chat UI backed by a **FastAPI** backend (`frontend/` and `backend/` at the project root), not a Streamlit app. It includes:

- **Real per-user authentication** — register/login backed by Amazon Cognito (the same `MCPServerPool` the Runtime/Gateway JWT authorizers trust), not a single shared demo credential
- **Multi-conversation chat interface** — a sidebar lists every conversation, each with its own AgentCore Memory session, and conversations can be deleted individually
- **Per-user conversation history** — persisted locally, namespaced per logged-in username, so multiple people sharing a browser never see each other's chats
- **Response timing** — surfaced per message for transparency

### Architecture for Lab 6

The frontend/backend connect to the AgentCore Runtime endpoint deployed in Lab 4, providing a complete end-to-end customer support solution. See the full system architecture (including the frontend/backend's place in it) in the project's root [README.md](../../README.md#architecture):

<div style="text-align:left">
    <img src="images/architecture_lab6_frontend.png" width="100%"/>
</div>

### What You'll learn

- How authentication is implemented end-to-end with Amazon Cognito (MCPServerPool)
- How the frontend manages multiple conversations and maps them to AgentCore Memory sessions
- How to run the full stack locally (Docker Compose or natively)
- How to exercise the complete customer journey from registration to support resolution

### Lab Objectives

By the end of this lab, you will have:

- Run the customer-facing Next.js + FastAPI web application locally
- Registered a user and logged in via Amazon Cognito (MCPServerPool)
- Used the multi-conversation chat interface, including deleting a conversation
- Verified memory continuity across messages and across page reloads
- Tested the full customer journey from login to support resolution

## Prerequisites

- **Completed Labs 1-5**
- **Docker & Docker Compose** (recommended way to run both services together), or **Node.js >= 18** and **Python 3.12+** to run them natively
- **AgentCore Runtime endpoint** from Lab 4 (deployed and ready)
- **Amazon Cognito `MCPServerPool`** configured for authentication (Lab 3 / `iac/modules/cognito_lab`)

### Step 1: Verify Prerequisites

The frontend and backend read every AWS resource identifier they need (Runtime ARN, MCPServerPool pool/client ID, discovery URL) from SSM at startup — the same parameters published by Terraform that previous labs have been reading. Let's confirm they're in place before launching anything.

In [ ]:
from lab_helpers.utils import get_ssm_parameter

required_parameters = [
    "/app/customersupport/agentcore/runtime_arn",
    "/app/customersupport/agentcore/mcp_lab/pool_id",
    "/app/customersupport/agentcore/mcp_lab/client_id",
    "/app/customersupport/agentcore/mcp_lab/cognito_discovery_url",
]

for name in required_parameters:
    value = get_ssm_parameter(name, with_decryption=False)
    print(f"✅ {name} -> {value}")

print("\nAll required SSM parameters are in place — the frontend/backend are ready to run.")

### Step 2: Understanding the Frontend Architecture

The application is split into two standalone services, each with its own README covering setup, configuration, and an architecture walkthrough:

- **[`frontend/`](../../frontend/README.md)** — Next.js 15 (App Router) + TypeScript + Tailwind CSS
- **[`backend/`](../../backend/README.md)** — FastAPI, layered into `core/` (config, logging), `api/routes/` (HTTP endpoints), `models/` (Pydantic schemas), `services/` (Cognito + AgentCore integrations), and `utils/`. Interactive API docs are auto-generated at `/docs` (Swagger UI) once it's running.

#### Core Components:

**Frontend** (`frontend/`)
1. `app/page.tsx` — auth gate: shows the login/register screen or the chat app
2. `components/AuthForm.tsx` — combined login/register screen
3. `components/ChatApp.tsx` — top-level chat orchestrator (conversations, sending messages)
4. `components/Sidebar.tsx` — conversation list, new chat, delete (with a confirmation dialog), logout
5. `lib/auth.ts` / `lib/conversations.ts` — session and per-user conversation persistence

**Backend** (`backend/`)
1. `api/routes/auth.py` — `/api/auth/register`, `/login`, `/refresh`
2. `api/routes/chat.py` — `/api/chat`, proxies messages to the AgentCore Runtime
3. `services/cognito_auth.py` — Cognito integration (per-user SRP auth + the shared service-identity fallback)
4. `services/agentcore_client.py` — invokes the Runtime's data-plane HTTP API directly (no `.bedrock_agentcore.yaml` dependency, same approach used in Lab 4/5's notebook cells)

#### Authentication Flow:

1. The user registers or logs in through the frontend's `AuthForm`
2. The backend authenticates against `MCPServerPool` (register: `admin_create_user` + `admin_set_user_password`; login: SRP via `pycognito`) — the **same pool** the Runtime/Gateway `CUSTOM_JWT` authorizers trust
3. The resulting access token is kept in the browser's `sessionStorage` and sent with every chat message, refreshed automatically before it expires
4. The backend forwards that token as the AgentCore Runtime's `Authorization` header, and uses the username as the AgentCore Memory `actor_id` — so memory and personalization are tied to the real logged-in user, not a shared demo identity

### Step 3: Launch the Customer Support Frontend 🚀

The recommended way to run both services together is **Docker Compose**, from the project root (one-time setup: copy `.env.example` to `.env` at the project root and set `AWS_PROFILE`/`AWS_REGION` — see the root [README.md](../../README.md#run-with-docker-compose)):

```bash
docker compose up --build
```

This builds and starts:
- **Backend** at `http://localhost:8000` (Swagger UI at `http://localhost:8000/docs`)
- **Frontend** at `http://localhost:3000`

Prefer running natively instead of Docker? Open two terminals:

```bash
# Terminal 1 — backend
cd backend
python3 -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
uvicorn app.main:app --reload --port 8000
```

```bash
# Terminal 2 — frontend
cd frontend
npm install
cp .env.local.example .env.local
npm run dev
```

**Important Notes:**
- Both services run continuously until you stop them (Ctrl+C, or `docker compose down` in another terminal)
- Make sure your AgentCore Runtime from Lab 4 is still deployed and running
- Cognito access tokens are short-lived (60 minutes) but the frontend refreshes them automatically while you're using the app

### Step 4: Testing Your Customer Support Application

Once both services are running, open `http://localhost:3000` and test the complete customer support experience:

#### Authentication Testing:
1. **Register a new account** — pick any username (3+ characters) and password (8+ characters); email is optional. This calls `/api/auth/register`, which creates a real user in `MCPServerPool` and logs you in immediately
2. **Log out and log back in** with the same credentials to verify `/api/auth/login` works independently of registration
3. **Verify** you land on the chat screen with your username shown in the sidebar footer

#### Customer Support Scenarios to Test:

- *"What's your return policy for laptops?"*
- *"I have a Gaming Console Pro, serial MNO33333333 — is it still under warranty?"*
- *"My laptop keeps overheating and restarting, what should I check?"*
- *"Search the web for the latest Samsung Galaxy S24 specs"*

#### Multi-Conversation & Memory Testing:

- Click **"New chat"** to start a second, independent conversation (its own AgentCore Memory session)
- Switch between conversations in the sidebar — each keeps its own message history
- Hover a conversation and click the trash icon to **delete** it (confirms via a dialog first) — this only removes it from the sidebar, not the underlying AgentCore Memory session
- Have a conversation, then **reload the page** — you should land back on the same conversation list, still logged in (session persists in `sessionStorage` until the tab closes), with full message history intact

#### What to Observe:

- **Response timing** — performance metrics displayed with each response
- **Memory persistence** — the agent remembers conversation context within a conversation
- **Tool integration** — the agent uses appropriate tools for different queries (warranty lookup, web search, return policy, technical support)
- **Per-user isolation** — log in as a different user (or open an incognito window) and confirm you don't see the first user's conversations
- **Error handling** — graceful handling of any issues (e.g. try logging in with a wrong password)

## 🎉 Lab 6 Complete!

Congratulations! You've explored a complete customer-facing frontend application for your AI-powered Customer Support Agent. Here's what you saw:

### What You Used

- **Web Interface** — Next.js + Tailwind CSS customer support application
- **Secure Authentication** — real per-user register/login via Amazon Cognito (`MCPServerPool`), not a single shared demo credential
- **Multi-Conversation Sidebar** — independent AgentCore Memory sessions per conversation, with delete support
- **Documented API** — a FastAPI backend with interactive Swagger docs (`/docs`)
- **Complete Integration** — frontend → backend → AgentCore Runtime, end to end

### End-to-End Customer Support Solution

You now have a **complete, customer support system** that includes:

1. **Intelligent Agent** (Lab 1) - AI-powered support with custom tools
2. **Persistent Memory** (Lab 2) - Conversation context and personalization
3. **Shared Tools & Identity** (Lab 3) - Scalable tool sharing and access control
4. **Production Runtime** (Lab 4) - Secure, scalable deployment with observability
5. **Online Evaluation** (Lab 5) - Continuous quality monitoring
6. **Customer Frontend** (Lab 6) - a real, authenticated web interface for end users

### Key Capabilities Demonstrated

- **Multi-turn Conversations** - Agent maintains context across interactions, across multiple independent conversations
- **Tool Integration** - Seamless use of product info, return policy, warranty lookup, and web search
- **Memory Persistence** - Customer preferences and history maintained per user, per conversation
- **Security & Identity** - Real per-user authentication and authorization, not a shared demo identity
- **Observability** - Full tracing and monitoring of agent behavior, plus the backend's own Swagger-documented API

### Next Steps

To further enhance your customer support solution, consider:

- **Custom Styling** - Brand the frontend with your company's design system
- **Additional Tools** - Integrate with your existing CRM, ticketing, or knowledge base systems
- **Multi-language Support** - Add internationalization for global customers
- **Advanced Analytics** - Implement custom dashboards for support team insights
- **Mobile Optimization** - Ensure the interface works well on mobile devices

### Cleanup

When you're ready to clean up the resources created in this workshop:

**Ready to clean up?** [Proceed to Lab 7: Cleanup →](lab-07-cleanup.ipynb)

---

**🎊 Congratulations on completing the Amazon Bedrock AgentCore End-to-End Workshop!**

You've successfully built a complete, production-ready AI agent solution from prototype to customer-facing application using Amazon Bedrock AgentCore capabilities.